# HD 206893 with amigo 0.1 and the 0.0.10 calibration
HD 206893 and its calibrator HD 205827 (JWST GO 1843, 2022-10-26, one exposure each in F380M, F430M and F480M),
from calslopes to companion fits, with `amigo-pipeline` and Virgil. The instrument is the paper's `calibration.npy`
(amigo 0.0.10). HD 206893 has two companions: the brown dwarf B (~200 mas) and the planet c (~100 mas).

Run with the venv `/fred/oz440/max/venvs/amigo-0.1` on a GPU (A100).

In [1]:
import json
import time
from pathlib import Path

import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as np
import matplotlib.pyplot as plt
import numpy as onp
import pandas as pd
import scipy.optimize as so
import dLux.utils as dlu
from IPython.display import Image, display

import amigo.files
from amigo.core_models import AmigoModel
from amigo.discos import to_virgil
from amigo.model_fits import SplineVisFit
from amigo.optical_models import AMIOptics
from amigo.vis_models import LogVisModel
from amigo_pipeline import Pipeline
from amigo_pipeline.calibration import load_calibration
from amigo_pipeline.dataset import DEFAULT_OPTICS, visibility_key
from virgil.grid_fit import likelihood_grid
from virgil.likelihood import model_loglike, whitened_residuals
from virgil.models import PointSource, System
from virgil.pipeline import BinaryPipeline

ERROR:2026-11-12 15:02:02,935:jax._src.xla_bridge:475: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/mcharles/.conda/envs/retrain/lib/python3.12/site-packages/jax/_src/xla_bridge.py", line 473, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/mcharles/.conda/envs/retrain/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/mcharles/.conda/envs/retrain/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: CUDA_ERROR_NO_DEVICE


ModuleNotFoundError: No module named 'amigo.discos'

## Data
Calslopes remade with amigo 0.1's `amigo.pipelines.process_calslope` (`bash/reprocess_calslope_amigo01.sh`).

In [ ]:
data_dir = "/fred/oz440/max/data/JWST/1843/calslope_amigo-0.1/"
files = amigo.files.get_files(data_dir, "calslope", EXP_TYPE="NIS_AMI", TARGPROP=["HD-206893", "HD-205827"])
files = sorted(files, key=lambda f: f.filename())

## Instrument calibration
The paper's `calibration.npy`, converted by `convert_calibration_v2.py` with per-visit piston/tip/tilt shared across
filters (`ptt` sharing `program`, which amigo `be7cee0` and later require).

In [ ]:
calibration = Path("/fred/oz440/max/data/amigo_files/v_0.0.11/v2_from_calibration_npy_ptt_program")
(detector, ramp, read), calibrated, defocus = load_calibration(calibration)

## Model
Exposures of one star, filter and roll share their visibilities.

In [ ]:
groups = {}
exposures = []
for f in files:
    role = "calibrator" if f[0].header["IS_PSF"] else "science"
    vis_key = visibility_key(f[0].header, role, groups, 1e-3)
    exposures.append(SplineVisFit(f, use_cov=True, only_diag=False, joint_fit=True, vis_key=vis_key))

filters = sorted({exp.filter for exp in exposures})
optics = AMIOptics(calibrated=calibrated, filters=filters, **DEFAULT_OPTICS)
placeholder = LogVisModel(dict(otf_coords=dlu.pixel_coords(3, 2 * optics.diameter),
                               modes={f: np.ones((1, 4), dtype=complex) / 2 for f in filters}))
model = AmigoModel(exposures, optics, detector, ramp, read, vis_model=placeholder)
for f in filters:
    model = model.set(f"params.defocus.{f}", np.asarray(defocus[f]))

## Pairs
Each HD 206893 exposure is calibrated by the HD 205827 exposure in its filter.

In [ ]:
pairs = []
for f in filters:
    calibrators = sorted(exp.key for exp in exposures if exp.filter == f and exp.calibrator)
    visits = {}
    for exp in exposures:
        if exp.filter == f and not exp.calibrator:
            visits.setdefault(exp.vis_key, []).append(exp)
    for visit in visits.values():
        pairs.append(dict(name=f"{visit[0].star}_{f}_obs{visit[0].observation}",
                          science=[exp.key for exp in visit], calibrator=calibrators, reference=calibrators[0]))

## Run the pipeline
`run()` resumes: stages already completed in `output` with the same inputs are loaded, not refitted, and a stage
whose inputs differ raises instead of being overwritten. `output` below is the 2026-11-11 run, so this takes
minutes. A new folder runs everything (about 1.6 h and 17 GB on an A100).

In [ ]:
fits, null = {"visibility": {}}, None
# BFGS's own stopping tolerances (1e-6) end the visibility fits before the gradient target
fits["visibility"].update(rtol=1e-8, atol=1e-8)
# kept from AB Dor, whose group 01093_013_02_03_1 stalls at a gradient of 1.8e-6 (2026-11-03)
fits["visibility"].update(tolerance=5e-6)
output = Path("/fred/oz440/max/outputs/retrain/pipeline/26-11-11_hd206893_v040")
pipeline = Pipeline(exposures, model, pairs, output=output, fits=fits, null=null)
discos, cross = pipeline.run()

The pipeline draws its own figures: the calibrator's instrument fit, HD 206893 fitted as a point source (the
companions show in the residuals), and the DISCOs.

In [ ]:
for figure in ["instrument/pixels_0.png", "reference/pixels_0.png", "products/discos.png"]:
    display(Image(filename=str(output / figure)))

## B: one companion
Virgil's binary pipeline on each filter's DISCOs. It fits a single companion, so it finds B, the brighter one.

The fit's reduced χ² is 3-5 with the errors as quoted, so the quoted errors are too small; the `x √χ²` columns scale
them up to make χ²/dof = 1 (a rough correction: it assumes the excess scatter is white).

PA here is east of north (Virgil's convention). The AMIGO paper's tables give 90° − PA (the angle from east
towards north): its AB Dor C 169.48° is 280.52° east of north, and our pipeline gives 280.55°. GRAVITY puts B at
PA 33° in 2019-07 and 6° in 2021-08 (Hinkley et al. 2023, Table 2), moving about −13° a year, so about 350° at
this epoch.

In [ ]:
results = {name: BinaryPipeline(to_virgil(disco), output=output / "binary" / name, max_sep_mas=600.0, n_flux=80).run()
           for name, disco in discos.items()}

rows = {}
for name, res in results.items():
    c, s = res.summary["companion"], res.summary["chi2"]
    k = onp.sqrt(s["companion_reduced"])
    rows[name.split("_")[1]] = dict(sep_mas=c["sep_mas"], sep_err=c["sep_err_mas"], sep_err_x_sqrt_chi2=k * c["sep_err_mas"],
                                    pa_deg=c["pa_deg"], pa_err=c["pa_err_deg"], pa_err_x_sqrt_chi2=k * c["pa_err_deg"],
                                    delta_mag=c["delta_mag"], chi2_reduced=s["companion_reduced"],
                                    detection_sigma=res.summary["search"]["local_nsigma"])
B_table = pd.DataFrame(rows).T
B_table.round(2)

The AMIGO paper's B (Table 6, per-filter fits), with its PA converted to east of north:

In [ ]:
paper_B = pd.DataFrame({"F380M": (199.3, 101.9, 8.33), "F430M": (192.1, 99.4, 7.96), "F480M": (199.4, 104.0, 7.47)},
                       index=["sep_mas", "paper_pa", "delta_mag"]).T
paper_B["pa_deg (east of north)"] = (90 - paper_B["paper_pa"]) % 360
paper_B.join(B_table[["sep_mas", "pa_deg", "delta_mag"]], rsuffix=" (this run)").round(2)

In [ ]:
plots = output / "binary" / next(iter(results)) / "plots"
for plot in ["search_delta_chi2", "limits_contrast_curve", "posterior_corner", "fit_correlation"]:
    display(Image(filename=str(plots / f"{plot}.png")))

## c: a second companion with B in the model
B held at its fit above, plus a second point source on a grid of positions and fluxes. The map is the
improvement in χ² over B alone, at the best flux in each position. The circle marks the paper's c (Table 7, per
filter, PA converted); the cross marks B.

In [ ]:
axis = onp.arange(-300.0, 301.0, 10.0)   # mas
fluxes = onp.geomspace(1e-5, 3e-3, 30)  # c / A
paper_c = {"F380M": (125, -88.0), "F430M": (102, -79.6), "F480M": (91, -83.8)}  # sep (mas), paper PA (deg)


def offsets(sep, pa_deg):
    th = onp.radians(pa_deg)
    return sep * onp.sin(th), sep * onp.cos(th)


searches = {}
for name, disco in discos.items():
    data = to_virgil(disco)
    b = results[name].summary["companion"]
    dra, ddec = offsets(b["sep_mas"], b["pa_deg"])
    B = PointSource(flux=b["flux"], dra=dra, ddec=ddec)
    ll_B = float(model_loglike(System(star=PointSource(), B=B), data))
    scene = System(star=PointSource(), B=B, c=PointSource(flux=1e-4, dra=0.0, ddec=0.0))
    ll = onp.asarray(likelihood_grid(scene, data=data, grid={"c.dra": axis, "c.ddec": axis, "c.flux": fluxes}))
    dchi2 = 2 * (ll.max(-1) - ll_B)  # (dra, ddec)
    best_flux = fluxes[ll.argmax(-1)]
    # two starting points for the joint fit: the map's peak, and its peak inside 150 mas (where GRAVITY's orbit puts c)
    starts = []
    for inside in (onp.ones_like(dchi2, bool), onp.hypot(*onp.meshgrid(axis, axis, indexing="ij")) <= 150):
        i, j = onp.unravel_index(onp.argmax(onp.where(inside, dchi2, -onp.inf)), dchi2.shape)
        starts.append((axis[i], axis[j], best_flux[i, j]))
    searches[name] = dict(data=data, B=(dra, ddec, b["flux"]), ll_B=ll_B, dchi2=dchi2, starts=starts)

fig, axes = plt.subplots(1, len(searches), figsize=(5 * len(searches), 4.4))
for ax, (name, s) in zip(onp.atleast_1d(axes), searches.items()):
    filt = name.split("_")[1]
    m = ax.imshow(s["dchi2"].T, origin="lower", cmap="inferno", vmin=0,
                  extent=[axis[0] - 5, axis[-1] + 5, axis[0] - 5, axis[-1] + 5])
    ax.invert_xaxis()  # east to the left
    sep, pa = paper_c[filt]
    ax.plot(*offsets(sep, 90 - pa), "o", mfc="none", mec="c", ms=14, label="paper c")
    ax.plot(*s["B"][:2], "x", c="w", ms=10, label="B (held)")
    ax.plot(0, 0, "*", c="w", ms=10)
    ax.set(title=f"{filt}: peak Δχ² {s['dchi2'].max():.0f}", xlabel="ΔRA (mas)", ylabel="ΔDec (mas)")
    plt.colorbar(m, ax=ax, fraction=0.046, label="Δχ² over B alone")
onp.atleast_1d(axes)[0].legend(loc="lower left", fontsize=8)
fig.tight_layout()
fig.savefig(output / "binary" / "c_search.png", dpi=110)
plt.close(fig)
display(Image(filename=str(output / "binary" / "c_search.png")))

## B and c fitted together
Both companions free (position and log flux), with B starting from its fit above and c from two places: the map's
peak, and its peak inside 150 mas. The better fit is kept; `dchi2_other_start` is the other one's, so a near-tie
shows. (F480M on 2026-11-12: the southern start gives Δχ² 76.0 with B back at 199 mas; the map's peak gives 75.2
with B pulled in to 173 mas and "c" at 223 mas, PA 264°.) Errors come from the curvature at the best fit, quoted
and scaled by √χ²/dof as for B. Inside the resolution limit (λ/2B ≈ 60-75 mas), separation and contrast trade off,
so a c that comes out close also comes out bright.

In [ ]:
def joint_fit(s, start):
    data = s["data"]

    def build(x):
        return System(star=PointSource(), B=PointSource(flux=np.exp(x[2]), dra=x[0], ddec=x[1]),
                      c=PointSource(flux=np.exp(x[5]), dra=x[3], ddec=x[4]))

    nll = jax.jit(lambda x: -model_loglike(build(x), data))
    grad = jax.jit(jax.grad(lambda x: -model_loglike(build(x), data)))
    (bx, by, bf), (cx, cy, cf) = s["B"], start
    r = so.minimize(lambda x: float(nll(x)), onp.array([bx, by, onp.log(bf), cx, cy, onp.log(cf)]),
                    jac=lambda x: onp.asarray(grad(x)), method="BFGS")
    cov = onp.linalg.inv(onp.asarray(jax.hessian(lambda x: -model_loglike(build(x), data))(r.x)))
    chi2 = float(onp.sum(onp.asarray(whitened_residuals(build(r.x), data)) ** 2))
    return r, cov, chi2


def polar(x, cov, k):
    dra, ddec = x[k], x[k + 1]
    sep, pa = onp.hypot(dra, ddec), onp.degrees(onp.arctan2(dra, ddec)) % 360
    J = onp.array([[dra / sep, ddec / sep], [onp.degrees(ddec / sep**2), -onp.degrees(dra / sep**2)]])
    C = J @ cov[k:k + 2, k:k + 2] @ J.T
    return sep, onp.sqrt(C[0, 0]), pa, onp.sqrt(C[1, 1]), -2.5 * x[k + 2] / onp.log(10), 2.5 / onp.log(10) * onp.sqrt(cov[k + 2, k + 2])


rows = {}
for name, s in searches.items():
    fits_ = sorted((joint_fit(s, start) for start in dict.fromkeys(s["starts"])), key=lambda f: f[0].fun)
    (r, cov, chi2), other = fits_[0], (2 * (-fits_[1][0].fun - s["ll_B"]) if len(fits_) > 1 else onp.nan)
    chi2_red = chi2 / (results[name].summary["chi2"]["n_independent"] - 6)
    k2 = onp.sqrt(chi2_red)
    for lab, k in [("B", 0), ("c", 3)]:
        sep, dsep, pa, dpa, dmag, ddmag = polar(r.x, cov, k)
        rows[(name.split("_")[1], lab)] = dict(sep_mas=sep, sep_err=dsep, sep_err_x_sqrt_chi2=k2 * dsep, pa_deg=pa, pa_err=dpa,
                                               pa_err_x_sqrt_chi2=k2 * dpa, delta_mag=dmag, dmag_err=ddmag,
                                               chi2_reduced=chi2_red, dchi2_over_B_only=2 * (-r.fun - s["ll_B"]),
                                               dchi2_other_start=other)
pd.DataFrame(rows).T.round(2)